# Nanoose Bay Pollution Control Centre

In [1]:
import datetime as dt
import gsw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import xarray as xr
import PyCO2SYS as pyco2

In [2]:
j=525
i=191

## Temperature (deg C)

In [4]:
temp_2025 = np.array([11.2, 11.2, 11.5, 13.0, 15.3, 17.2, 19.0, 19.6, 19.4, 17.0, 14.6, 13.0])
temp = temp_2025
temp

array([11.2, 11.2, 11.5, 13. , 15.3, 17.2, 19. , 19.6, 19.4, 17. , 14.6,
       13. ])

In [5]:
# Conservative temperature 
temp_conservative = gsw.CT_from_pt(0, temp)
print(temp_conservative)

[11.81200223 11.81200223 12.12712279 13.70225322 16.11608272 18.10904661
 19.99636641 20.62533014 20.41568298 17.89930153 15.38160007 13.70225322]


## Flux (m^3/day)

In [6]:
flux_m3d_2025 = np.array([391.5, 412.3, 449.3, 331.6, 280.2, 273.2, 294.6, 292.2, 278.7, 319.3, 345.9, 451.6])
flux_m3d = flux_m3d_2025
flux_m3d

array([391.5, 412.3, 449.3, 331.6, 280.2, 273.2, 294.6, 292.2, 278.7,
       319.3, 345.9, 451.6])

In [7]:
mesh_mask_nc = "/ocean/atall/MOAD/grid/mesh_mask_202310b.nc"
ds_mask = xr.open_dataset(mesh_mask_nc)
rho = 1000
seconds_per_day = 86400
cell_area = (ds_mask["e1t"].isel(t=0).values[j, i]* ds_mask["e2t"].isel(t=0).values[j, i])
new_flux = (flux_m3d * rho/ (cell_area * seconds_per_day))
print("New flux:", new_flux)
print("New flux is in kg/m2/S")

New flux: [2.11060905e-05 2.22274358e-05 2.42221365e-05 1.78768317e-05
 1.51058149e-05 1.47284391e-05 1.58821309e-05 1.57527449e-05
 1.50249487e-05 1.72137285e-05 1.86477566e-05 2.43461314e-05]
New flux is in kg/m2/S


## Ammonia (mg/L)

In [8]:
ammonia_2025 = np.array([33.30, 39.90, 32.30, 41.06, 37.85, 52.90, 52.20, 52.55, 51.47, 51.72, 44.47, 37.60])
ammonia = ammonia_2025
ammonia

array([33.3 , 39.9 , 32.3 , 41.06, 37.85, 52.9 , 52.2 , 52.55, 51.47,
       51.72, 44.47, 37.6 ])

In [9]:
# Convert mg/L NH3 to mmol/m³
molar_mass_NH3 = 17.031  # g/mol
ammonia_mmol_m3 = ammonia * 1000 / molar_mass_NH3
print(ammonia_mmol_m3)

[1955.25805883 2342.78668311 1896.54160061 2410.89777465 2222.41794375
 3106.10064001 3064.99911925 3085.54987963 3022.13610475 3036.81521931
 2611.12089719 2207.73882919]


## Alkalinity (mmol/m^3)

In [10]:
alkalinity = np.zeros(12)

## BOD (mg/L)

In [11]:
bod_2025 = np.array([52.2, 77.5, 60.1, 82.0, 79.5, 98.1, 97.3, 99.3, 76.6, 97.0, 71.9, 59.6])
bod = bod_2025
bod

array([52.2, 77.5, 60.1, 82. , 79.5, 98.1, 97.3, 99.3, 76.6, 97. , 71.9,
       59.6])

## PON and DON

In [12]:
# Convert BOD from mg O2/L to mmol O2/m3,then estimate biodegradable organic carbon in mmol C/m3
boc_mmol_m3 = bod * 1000 / 32 * 106 / 138
# Assumed molar carbon-to-nitrogen ratio
carbon_to_nitrogen_ratio = 34
# Assumption: DOC = 0.8 * POC
doc_to_poc_ratio = 0.8
# Split biodegradable organic carbon into particulate and dissolved carbon
poc_mmol_m3 = boc_mmol_m3 / (1 + doc_to_poc_ratio)
doc_mmol_m3 = poc_mmol_m3 * doc_to_poc_ratio
# Convert particulate and dissolved carbon to organic nitrogen
PON = poc_mmol_m3 / carbon_to_nitrogen_ratio
DON = doc_mmol_m3 / carbon_to_nitrogen_ratio

print("Estimated PON in mmol N/m3:")
print(PON)

print("\nEstimated DON in mmol N/m3:")
print(DON)

Estimated PON in mmol N/m3:
[20.4736786  30.39674505 23.57218552 32.16171734 31.18117718 38.47639599
 38.16262314 38.94705527 30.04375059 38.04495832 28.20033509 23.37607748]

Estimated DON in mmol N/m3:
[16.37894288 24.31739604 18.85774841 25.72937388 24.94494174 30.78111679
 30.53009851 31.15764422 24.03500047 30.43596666 22.56026807 18.70086199]


## pH

In [13]:
ph_2025 = np.array([7.19, 7.28, 7.36, 7.41, 7.34, 7.31, 7.29, 7.23, 7.30, 7.34, 7.33, 7.20])
ph = ph_2025
ph

array([7.19, 7.28, 7.36, 7.41, 7.34, 7.31, 7.29, 7.23, 7.3 , 7.34, 7.33,
       7.2 ])

## DIC

In [ ]:
dic = np.zeros(12)

## TSS (mg/L)

In [15]:
tss_2025 = np.array([26.9, 28.9, 34.5, 40.8, 32.6, 30.1, 35.8, 30.0, 43.3, 42.8, 32.1, 36.7])
tss = tss_2025
tss

array([26.9, 28.9, 34.5, 40.8, 32.6, 30.1, 35.8, 30. , 43.3, 42.8, 32.1,
       36.7])

## Turbidity

In [16]:
turbidity_fau = (tss - 7) / 0.89
print("Turbidity in FAU:")
print(turbidity_fau)

Turbidity in FAU:
[22.35955056 24.60674157 30.8988764  37.97752809 28.76404494 25.95505618
 32.35955056 25.84269663 40.78651685 40.2247191  28.20224719 33.37078652]


## Nitrate 

In [17]:
nitrate = np.zeros(12)

## Oxygen (mg/L)

In [18]:
oxygen_2025 = np.array([3.42, 4.12, 4.64, 2.68, 2.19, 2.29, 2.02, 2.45, 2.17, 2.63, 2.29, 3.29])
oxygen = oxygen_2025
oxygen

array([3.42, 4.12, 4.64, 2.68, 2.19, 2.29, 2.02, 2.45, 2.17, 2.63, 2.29,
       3.29])

In [19]:
oxygen_mmol_m3 = oxygen * 1000 / 32.0
oxygen_mmol_m3

array([106.875 , 128.75  , 145.    ,  83.75  ,  68.4375,  71.5625,
        63.125 ,  76.5625,  67.8125,  82.1875,  71.5625, 102.8125])

In [20]:
dSi = np.zeros(12)
diatoms = np.zeros(12)
nanoflagellates = np.zeros(12)
z1 = np.zeros(12)
bSi = np.zeros(12)